# Module 6: MCMC Foundations for Bayesian Commodity Models

## Learning Objectives
By completing this notebook, you will:
1. Understand Markov Chain Monte Carlo (MCMC) as a posterior sampling strategy
2. Implement Metropolis-Hastings algorithm from scratch
3. Diagnose convergence using trace plots, autocorrelation, and R-hat
4. Apply MCMC to commodity price models with complex posteriors
5. Understand why advanced samplers (HMC/NUTS) are needed for high dimensions

## Prerequisites
- Bayesian inference fundamentals
- Probability distributions
- Python programming

## Estimated Time: 70 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats

np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"NumPy version: {np.__version__}")
print(f"PyMC version: {pm.__version__}")

## Conceptual Introduction

### The Posterior Sampling Problem

**Goal:** Draw samples from the posterior $p(\theta | y)$

**Challenge:** For most models, $p(\theta | y)$ has no closed form. We can only evaluate it up to a normalizing constant:

$$p(\theta | y) = \frac{p(y | \theta) p(\theta)}{\int p(y | \theta') p(\theta') d\theta'}$$

The denominator (marginal likelihood) is often intractable.

### MCMC Solution

**Key Insight:** Instead of computing $p(\theta | y)$ directly, construct a Markov chain whose stationary distribution IS the posterior.

**Markov Chain:** Sequence of states where next state depends only on current state:
$$\theta^{(t+1)} \sim K(\theta^{(t+1)} | \theta^{(t)})$$

**Stationary Distribution:** After "burn-in", samples $\theta^{(t)} \sim p(\theta | y)$

### Metropolis-Hastings Algorithm

1. Start at $\theta^{(0)}$
2. Propose $\theta^* \sim q(\theta^* | \theta^{(t)})$ (proposal distribution)
3. Compute acceptance ratio:
   $$r = \frac{p(\theta^* | y)}{p(\theta^{(t)} | y)} \cdot \frac{q(\theta^{(t)} | \theta^*)}{q(\theta^* | \theta^{(t)})}$$
4. Accept with probability $\min(1, r)$:
   - If accept: $\theta^{(t+1)} = \theta^*$
   - If reject: $\theta^{(t+1)} = \theta^{(t)}$ (stay)
5. Repeat

### Why This Works

The acceptance ratio ensures **detailed balance**:
$$p(\theta) K(\theta' | \theta) = p(\theta') K(\theta | \theta')$$

This guarantees the chain converges to $p(\theta | y)$.

## Part 1: Metropolis-Hastings from Scratch

In [ ]:
# Purpose: Implement Metropolis-Hastings algorithm
# Key Concept: Accept/reject mechanism maintains detailed balance

def metropolis_hastings(log_posterior, theta_init, n_samples, proposal_sd):
    """
    Metropolis-Hastings MCMC sampler with symmetric Gaussian proposal.
    
    Parameters
    ----------
    log_posterior : callable
        Function computing log p(theta | y)
    theta_init : float or array
        Initial parameter value
    n_samples : int
        Number of MCMC iterations
    proposal_sd : float
        Standard deviation of Gaussian proposal
    
    Returns
    -------
    samples : array, shape (n_samples,)
        MCMC samples
    acceptance_rate : float
        Fraction of proposals accepted
    """
    # Initialize
    theta_current = theta_init
    log_p_current = log_posterior(theta_current)
    
    samples = np.zeros(n_samples)
    n_accepted = 0
    
    for i in range(n_samples):
        # Step 1: Propose new value (symmetric Gaussian random walk)
        theta_proposed = theta_current + np.random.normal(0, proposal_sd)
        log_p_proposed = log_posterior(theta_proposed)
        
        # Step 2: Compute acceptance ratio (log scale for numerical stability)
        # For symmetric proposal: q(theta'|theta) = q(theta|theta'), so ratio = 1
        log_ratio = log_p_proposed - log_p_current
        
        # Step 3: Accept or reject
        if np.log(np.random.uniform()) < log_ratio:
            # Accept
            theta_current = theta_proposed
            log_p_current = log_p_proposed
            n_accepted += 1
        # else: reject (theta_current stays the same)
        
        # Step 4: Store sample
        samples[i] = theta_current
    
    acceptance_rate = n_accepted / n_samples
    
    return samples, acceptance_rate

print("Metropolis-Hastings implementation complete.")
print("\nAlgorithm guarantees: Samples converge to target distribution p(θ|y)")

In [ ]:
# Purpose: Test on simple problem - Normal posterior
# Key Concept: Verify algorithm recovers known distribution

# Problem: Estimate mean of normal distribution
# Data: y ~ N(true_mu, sigma=2)
# Prior: mu ~ N(0, 10)
# Posterior: mu | y ~ N(posterior_mu, posterior_sigma)

# Generate data
true_mu = 5.0
sigma = 2.0
n_obs = 20
y_data = np.random.normal(true_mu, sigma, n_obs)

# Analytical posterior (conjugate)
prior_mu = 0.0
prior_sigma = 10.0
posterior_precision = 1/prior_sigma**2 + n_obs/sigma**2
posterior_sigma = 1/np.sqrt(posterior_precision)
posterior_mu = (prior_mu/prior_sigma**2 + y_data.sum()/sigma**2) / posterior_precision

print(f"Analytical posterior: N({posterior_mu:.3f}, {posterior_sigma:.3f})")

# Define log posterior
def log_posterior_normal(mu):
    # Log likelihood: sum of log N(y_i | mu, sigma)
    log_lik = -0.5 * np.sum((y_data - mu)**2) / sigma**2
    
    # Log prior: log N(mu | 0, 10)
    log_prior = -0.5 * (mu - prior_mu)**2 / prior_sigma**2
    
    return log_lik + log_prior

# Run Metropolis-Hastings
samples_mh, accept_rate = metropolis_hastings(
    log_posterior=log_posterior_normal,
    theta_init=0.0,
    n_samples=10000,
    proposal_sd=0.5
)

print(f"\nMCMC Results:")
print(f"Acceptance rate: {accept_rate:.1%}")
print(f"Sample mean: {samples_mh[1000:].mean():.3f}")
print(f"Sample std: {samples_mh[1000:].std():.3f}")
print(f"\nClose to analytical posterior? {np.abs(samples_mh[1000:].mean() - posterior_mu) < 0.1}")

In [ ]:
# Purpose: Visualize MCMC diagnostics
# Key Concept: Trace plots, autocorrelation, and posterior histogram

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Trace plot (shows mixing)
ax = axes[0, 0]
ax.plot(samples_mh, alpha=0.7, linewidth=0.5)
ax.axhline(posterior_mu, color='red', linestyle='--', linewidth=2, label='True Posterior Mean')
ax.axvspan(0, 1000, alpha=0.2, color='gray', label='Burn-in')
ax.set_xlabel('Iteration')
ax.set_ylabel('μ')
ax.set_title('Trace Plot (Should Look Like "Hairy Caterpillar")', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Running mean (convergence diagnostic)
ax = axes[0, 1]
running_mean = np.cumsum(samples_mh) / np.arange(1, len(samples_mh) + 1)
ax.plot(running_mean, linewidth=2)
ax.axhline(posterior_mu, color='red', linestyle='--', linewidth=2, label='Target')
ax.set_xlabel('Iteration')
ax.set_ylabel('Running Mean of μ')
ax.set_title('Convergence to Posterior Mean', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Autocorrelation (measures sample efficiency)
ax = axes[1, 0]
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(samples_mh[1000:], lags=50, ax=ax, alpha=0.05)
ax.set_title('Autocorrelation Function (Lower = More Independent)', fontsize=12, fontweight='bold')
ax.set_xlabel('Lag')

# Plot 4: Posterior distribution
ax = axes[1, 1]
ax.hist(samples_mh[1000:], bins=50, density=True, alpha=0.7, 
       edgecolor='black', label='MCMC Samples')

# Overlay analytical posterior
x_range = np.linspace(samples_mh.min(), samples_mh.max(), 200)
analytical_pdf = stats.norm.pdf(x_range, posterior_mu, posterior_sigma)
ax.plot(x_range, analytical_pdf, 'r-', linewidth=3, label='Analytical Posterior')

ax.set_xlabel('μ')
ax.set_ylabel('Density')
ax.set_title('Posterior Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nDiagnostic Interpretation:")
print("1. Trace plot: Should wander randomly (good mixing)")
print("2. Running mean: Should stabilize at true value")
print("3. Autocorrelation: Should decay quickly (efficient sampling)")
print("4. Histogram: Should match analytical distribution")

## Exercise 6.1: Tune Proposal Distribution

**Task:** Run Metropolis-Hastings with THREE different proposal standard deviations:
- Small (0.1): Proposals too conservative
- Optimal (~0.5): Good balance
- Large (5.0): Proposals too aggressive

Compare:
1. Acceptance rates
2. Effective sample size (ESS)
3. Trace plot mixing quality

**Expected Output:**
- Small SD: High acceptance (~95%), poor mixing, low ESS
- Optimal SD: Medium acceptance (~25-50%), good mixing, high ESS
- Large SD: Low acceptance (~5%), poor mixing, low ESS

**Hints:**
<details>
<summary>Hint 1</summary>
Use az.ess() to compute effective sample size
</details>

<details>
<summary>Hint 2</summary>
Rule of thumb: Target 20-40% acceptance rate
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Run MH with three proposal SDs
# 2. Compute acceptance rates and ESS
# 3. Create comparison plots

proposal_sds = [0.1, 0.5, 5.0]
results = {}

for sd in proposal_sds:
    # YOUR CODE: Run MH and collect diagnostics
    pass

your_answer = results  # Dictionary with results

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_1():
    assert your_answer is not None, "Implement your solution!"
    assert len(your_answer) == 3, "Need results for 3 different proposal SDs"
    
    # Optimal SD should have best ESS
    # (This is a simplified check - actual implementation would verify)
    
    print("Exercise 6.1 passed!")
    print("Optimal proposal balances exploration and acceptance")

test_exercise_6_1()

## Part 2: Apply to Commodity Price Model

### Stochastic Volatility Model

Commodity returns exhibit volatility clustering. Model:

$$r_t = \exp(h_t/2) \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, 1)$$
$$h_t = \mu + \phi (h_{t-1} - \mu) + \sigma_h \eta_t, \quad \eta_t \sim \mathcal{N}(0, 1)$$

Parameters: $(\mu, \phi, \sigma_h)$

**Challenge:** Latent variables $h_t$ make posterior complex (non-conjugate).

In [ ]:
# Purpose: Generate returns with stochastic volatility
# Key Concept: Volatility clustering (ARCH/GARCH effects)

n_t = 500
true_mu = -2.0  # Mean log-volatility
true_phi = 0.95  # Persistence
true_sigma_h = 0.3  # Volatility of volatility

# Simulate log-volatility process
h = np.zeros(n_t)
h[0] = true_mu
for t in range(1, n_t):
    h[t] = true_mu + true_phi * (h[t-1] - true_mu) + true_sigma_h * np.random.normal()

# Generate returns
returns = np.exp(h/2) * np.random.normal(size=n_t)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.plot(returns, linewidth=1, alpha=0.8)
ax.set_xlabel('Time')
ax.set_ylabel('Return (%)')
ax.set_title('Simulated Commodity Returns (Stochastic Volatility)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8)

ax = axes[1]
ax.plot(np.exp(h/2), linewidth=2, color='darkred')
ax.set_xlabel('Time')
ax.set_ylabel('Volatility')
ax.set_title('True Time-Varying Volatility', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"True parameters: μ={true_mu:.2f}, φ={true_phi:.2f}, σ_h={true_sigma_h:.2f}")
print("Note: Returns show clustering (calm periods, volatile periods)")

In [ ]:
# Purpose: Fit stochastic volatility model with PyMC
# Key Concept: NUTS handles complex posterior automatically

with pm.Model() as sv_model:
    # Priors on parameters
    mu = pm.Normal('mu', mu=-2, sigma=2)
    phi = pm.Uniform('phi', lower=-1, upper=1)  # Stationarity constraint
    sigma_h = pm.HalfNormal('sigma_h', sigma=1)
    
    # Initial log-volatility
    h_init = pm.Normal('h_init', mu=mu, sigma=sigma_h/np.sqrt(1-phi**2))
    
    # Log-volatility process (random walk component)
    h_innov = pm.GaussianRandomWalk(
        'h_innov',
        mu=0,
        sigma=sigma_h,
        shape=n_t-1
    )
    
    # Construct full h_t process
    h = pm.Deterministic(
        'h',
        pm.math.concatenate([
            [h_init],
            h_init + pm.math.cumsum(phi * h_innov)
        ])
    )
    
    # Likelihood: returns given volatility
    pm.Normal(
        'returns',
        mu=0,
        sigma=pm.math.exp(h/2),
        observed=returns
    )
    
    # Sample with NUTS (automatic tuning)
    trace_sv = pm.sample(
        1000,
        tune=1500,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

print("\nPosterior summary:")
print(az.summary(trace_sv, var_names=['mu', 'phi', 'sigma_h'], round_to=3))
print(f"\nTrue values: μ={true_mu}, φ={true_phi}, σ_h={true_sigma_h}")

## Summary

### Key Takeaways

1. **MCMC constructs chains** whose stationary distribution is the posterior

2. **Metropolis-Hastings** is simple but requires tuning (proposal distribution)

3. **Convergence diagnostics** are essential (trace plots, R-hat, ESS)

4. **Modern samplers (NUTS)** adaptively tune and handle complex geometries

5. **High-dimensional problems** require gradient-based methods (HMC/NUTS)

### What's Next

**Next Notebook: Hamiltonian Monte Carlo** - Gradient-based sampler for efficient high-dimensional exploration

**Module 7: Regime Switching** - Apply MCMC to models with discrete latent states

---

*"MCMC turns the intractable posterior into a traversable landscape."*